In [1]:
!pip install -q keras-tuner

In [2]:
import keras_tuner
import keras

In [4]:

import csv

# Path to the raw CSV file
csv_file_path = '/content/MovieReviewsData.csv'

data = []  # Initialize an empty list to store the data

with open(csv_file_path, 'r', encoding='utf-8', newline='') as csv_file:
    # Read the CSV file contents and convert them into tuples
    csv_data = csv.reader(csv_file)
    next(csv_data)  # Skip header if exists
    for row in csv_data:
        text = row[0]       # Assuming text is in the first column
        sentiment = row[1]  # Assuming sentiment is in the second column
        # Create a tuple with text and sentiment and append it to the data list
        data.append((text, sentiment))


In [5]:
# Import necessary libraries
import numpy as np
from keras.src.layers import Bidirectional  # Import Bidirectional layer from Keras
from keras_tuner import HyperModel  # Import HyperModel from Keras Tuner
from tensorflow.keras.models import Sequential  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D  # Import various layers from TensorFlow Keras
from tensorflow.keras.preprocessing.text import Tokenizer  # Import Tokenizer for text preprocessing
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Import pad_sequences for padding sequences
from tensorflow.keras.optimizers import Adam  # Import Adam optimizer
from tensorflow.keras.losses import SparseCategoricalCrossentropy  # Import SparseCategoricalCrossentropy loss
from sklearn.utils.class_weight import compute_class_weight  # Import compute_class_weight for calculating class weights
from sklearn.model_selection import train_test_split  # Import train_test_split for splitting data
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # Import callbacks for model training

from keras_tuner import BayesianOptimization
from tensorflow.keras.losses import SparseCategoricalCrossentropy  # Import SparseCategoricalCrossentropy loss from TensorFlow Keras
from nltk.corpus import stopwords  # Import stopwords from NLTK corpus
import re  # Import re module for regular expressions
import nltk  # Import NLTK library for natural language processing
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
# Preprocessing function to clean and tokenize text data
def preprocess(text):
    text = text.lower()  # Convert text to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stopwords.words('english')])  # Remove stopwords
    return text

# Apply preprocessing to the dataset
corpus = [preprocess(text) for text,_ in data]

# Tokenization and padding
tokenizer = Tokenizer(num_words=5000, oov_token='')  # Initialize Tokenizer with a vocabulary size limit
tokenizer.fit_on_texts(corpus)  # Fit Tokenizer on preprocessed text data
word_index = tokenizer.word_index  # Get word index from Tokenizer
sequences = tokenizer.texts_to_sequences(corpus)  # Convert text to sequences of indices
padded_sequences = pad_sequences(sequences, padding='post')  # Pad sequences to the same length

# Convert sentiment labels to numerical values
sentiment_mapping = {"positive": 0, "negative": 1, "neutral": 2,"neg": 1,"pos": 0,"neu":2}  # Mapping of sentiment labels to numerical values
labels = np.array([sentiment_mapping[sentiment] for _, sentiment in data])  # Convert sentiment labels to numerical values

In [7]:

# Split the dataset into training (70%), validation (15%) and test (15%)

from sklearn.model_selection import train_test_split

# First split: 70% training and 30% temporary data
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    padded_sequences,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Second split: Divide the remaining 30% equally
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

# Display dataset sizes
print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))
print("Testing samples:", len(test_texts))


# Calculate class weights using ONLY training labels
class_counts = np.bincount(train_labels)
total_samples = sum(class_counts)

class_weights = {
    cls: total_samples / count
    for cls, count in enumerate(class_counts)
}


# Define a custom hypermodel for Keras Tuner
class SentimentHyperModel(HyperModel):

    def build(self, hp):

        model = Sequential()

        model.add(Embedding(
            len(word_index) + 1,
            hp.Int('embedding_dim', min_value=64, max_value=256, step=32)
        ))

        model.add(Bidirectional(LSTM(
            hp.Int('lstm_units', min_value=64, max_value=128, step=32),
            return_sequences=True
        )))

        model.add(GlobalMaxPooling1D())

        model.add(Dense(
            hp.Int('dense_units', min_value=64, max_value=256, step=32),
            activation='relu'
        ))

        model.add(Dropout(
            hp.Float('dense_dropout', min_value=0.2, max_value=0.5, step=0.1)
        ))

        model.add(Dense(3, activation='softmax'))

        optimizer = Adam(
            learning_rate=hp.Float(
                'learning_rate',
                min_value=1e-4,
                max_value=1e-2,
                sampling='LOG'
            )
        )

        loss = SparseCategoricalCrossentropy()

        model.compile(
            loss=loss,
            optimizer=optimizer,
            metrics=['accuracy']
        )

        return model


# Instantiate the custom hypermodel
hypermodel = SentimentHyperModel()


Training samples: 17500
Validation samples: 3750
Testing samples: 3750


In [8]:
# Use Bayesian Optimisation for hyperparameter tuning
tuner = BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=5,
    num_initial_points=3,
    directory='my_dir',
    project_name='sentiment_analysis_bayesian'
)

# Perform hyperparameter search using training data
tuner.search(
    train_texts,
    train_labels,
    epochs=4,
    validation_data=(val_texts, val_labels),
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.2, patience=2)
    ]
)

# Load the best hyperparameters found by the tuner
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model using the best hyperparameters
best_model = hypermodel.build(best_hps)

print(best_model)


Trial 5 Complete [00h 04m 16s]
val_accuracy: 0.8829333186149597

Best val_accuracy So Far: 0.890666663646698
Total elapsed time: 00h 20m 08s
<Sequential name=sequential_1, built=False>


In [9]:

# Import evaluation metrics
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

# Train the best model with training data
best_model.fit(
    train_texts,
    train_labels,
    epochs=4,
    validation_data=(val_texts, val_labels),
    class_weight=class_weights
)

# Load the best model after tuning
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate the model using test data
test_probabilities = best_model.predict(test_texts)

# Convert predicted probabilities into class labels
y_pred = np.argmax(test_probabilities, axis=1)

# Calculate F1 scores
f1 = f1_score(test_labels, y_pred, average='weighted')
precision = precision_score(test_labels, y_pred, average='weighted')
recall = recall_score(test_labels, y_pred, average='weighted')
macro_f1 = f1_score(
    test_labels, y_pred, average='macro'
)
print("Test Precision:", precision)
print("Test Recall:", recall)
print("Weighted F1 Score:", f1)
print("Macro F1 Score:", macro_f1)

# Display detailed classification report
print("\nClassification Report:")
print(classification_report(test_labels, y_pred))



Epoch 1/4
547/547 ━━━━━━━━━━━━━━━━━━━━ 62s 108ms/step - accuracy: 0.7876 - loss: 0.8494 - val_accuracy: 0.8613 - val_loss: 0.3036
Epoch 2/4
547/547 ━━━━━━━━━━━━━━━━━━━━ 56s 102ms/step - accuracy: 0.9129 - loss: 0.4463 - val_accuracy: 0.8955 - val_loss: 0.2639
Epoch 3/4
547/547 ━━━━━━━━━━━━━━━━━━━━ 56s 102ms/step - accuracy: 0.9504 - loss: 0.2823 - val_accuracy: 0.8723 - val_loss: 0.3231
Epoch 4/4
547/547 ━━━━━━━━━━━━━━━━━━━━ 54s 99ms/step - accuracy: 0.9727 - loss: 0.1586 - val_accuracy: 0.8603 - val_loss: 0.3858


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 24 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step
Test Precision: 0.8829647009991708
Test Recall: 0.8810666666666667
Weighted F1 Score: 0.8809191206939714
Macro F1 Score: 0.8809191206939713

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.92      0.89      1875
           1       0.91      0.85      0.88      1875

    accuracy                           0.88      3750
   macro avg       0.88      0.88      0.88      3750
weighted avg       0.88      0.88      0.88      3750

